[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day10_lab.ipynb)

# Day 10 · 실습 — RAG — 내 문서로 답하게 하기

실제 법령 조문으로 찾아 읽고 근거로만 답한다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

실습 시간에 푼다. **강의 노트북에 없던 문제**들이다.

`스스로 풀기` 는 각자, `조별로 풀기` 는 2~3명이 한 조로 상의하며 푼다.
막히면 강의 노트북(`live`)에서 같은 함수를 쓴 셀을 찾아 대조한다.

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 준비

In [ ]:
# 문서 네 개를 받아 온다. 사내에서는 이 자리가 공유 폴더나 문서함이 된다.
import urllib.request, re, json

BASE = 'https://tunalee.github.io/posco/data/docs/'
FILES = {'근로기준법': 'labor_standards.txt', '산업안전보건법': 'occupational_safety.txt',
         '산업기술보호법': 'industrial_tech.txt', '개인정보보호법': 'privacy.txt'}

DOCS = {}
for name, fn in FILES.items():
    raw = urllib.request.urlopen(BASE + fn, timeout=60).read().decode('utf-8')
    DOCS[name] = '\n'.join(l for l in raw.split('\n') if not l.startswith('#'))   # 머리말 주석은 뺀다
    print('%-12s %6d자' % (name, len(DOCS[name])))

In [ ]:
# 키는 화면에 안 찍히게 받는다
import getpass
KEY = getpass.getpass('nvapi- 로 시작하는 키: ')

URL = 'https://integrate.api.nvidia.com/v1/chat/completions'
MODEL = 'nvidia/llama-3.3-nemotron-super-49b-v1'

def ask(prompt, n=500):
    body = json.dumps({'model': MODEL, 'max_tokens': n, 'temperature': 0,
                       'messages': [{'role': 'user', 'content': prompt}]}).encode()
    req = urllib.request.Request(URL, data=body, headers={
        'Authorization': 'Bearer ' + KEY,
        'Content-Type': 'application/json', 'Accept': 'application/json'})
    for _ in range(2):
        try:
            with urllib.request.urlopen(req, timeout=180) as f:
                return json.load(f)['choices'][0]['message']['content'].strip()
        except Exception as e:
            err = str(e)[:80]
    return '[실패] ' + err

## 2. 자르기

In [ ]:
# 자르면서 메타데이터를 같이 붙인다. 나중에 범위를 좁힐 때 쓴다.
def split_articles(text, source):
    out, chapter = [], ''
    for p in re.split(r'\n(?=제\d+조)', text):
        p = p.strip()
        ch = re.findall(r'^\[제\d+장[^\]]*\]', p, flags=re.M)
        if ch:
            chapter = ch[-1].strip('[]')      # 장이 바뀌면 갈아 끼운다
        if len(p) < 40:                       # 너무 짧은 조각은 버린다
            continue
        head = p.split('\n')[0]
        num = re.match(r'제(\d+)조', head)
        out.append({'source': source, 'chapter': chapter,
                    'article': int(num.group(1)) if num else 0,
                    'title': head[:40], 'text': p})
    return out

CHUNKS = []
for name, text in DOCS.items():
    CHUNKS += split_articles(text, name)
print('조각 %d개' % len(CHUNKS))
print('평균 %d자 · 가장 긴 것 %d자' % (sum(len(c['text']) for c in CHUNKS) / len(CHUNKS),
                                max(len(c['text']) for c in CHUNKS)))

## 3. 찾기

In [ ]:
# 낱말이 겹치는 정도로 순위를 매긴다 (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=50000)
M = vec.fit_transform([c['text'] for c in CHUNKS])

def find(question, k=3):
    sim = cosine_similarity(vec.transform([question]), M)[0]
    return [(sim[i], CHUNKS[i]) for i in sim.argsort()[::-1][:k]]

def show(question, k=3):
    print('Q', question)
    for s, c in find(question, k):
        print('  %.3f [%s] %s' % (s, c['source'], c['title']))
    print()

## 4. 의미로 찾기

In [ ]:
# 임베딩 모델을 받는다. 처음 한 번만 몇 분 걸린다.
%pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer
import numpy as np

EMB = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print('벡터 차원', EMB.get_sentence_embedding_dimension())

In [ ]:
# 조각 전부를 좌표로 바꿔 둔다. 이것이 인덱싱 타임이다.
TEXTS = [c['title'] + ' ' + c['text'][:400] for c in CHUNKS]

def build(model):
    return model.encode(TEXTS, batch_size=64, normalize_embeddings=True,
                        show_progress_bar=False)

V = build(EMB)
MULTI, VM = EMB, V          # 나중에 견주려고 다국어 판을 따로 남겨 둔다
print('조각 %d개 × %d차원' % V.shape)

In [ ]:
# 질문마다 도는 부분 — 이것이 쿼리 타임이다.
def find2(question, k=3, model=None, mat=None):
    model, mat = model or EMB, V if mat is None else mat
    q = model.encode([question], normalize_embeddings=True)[0]
    sim = mat @ q
    return [(sim[i], CHUNKS[i]) for i in sim.argsort()[::-1][:k]]

def show2(question, k=3):
    print('Q', question)
    for s, c in find2(question, k):
        print('  %.3f [%s] %s' % (s, c['source'], c['title']))
    print()

In [ ]:
# 질문마다 정답 조문을 못 박아 둔다. (문서, 조 번호) 로 적어야 인용만 한 조문이 안 걸린다.
QS = [('쉬는 날은 일 년에 며칠 받나', '근로기준법', '제60조'),
      ('휴가를 며칠이나 쓸 수 있나', '근로기준법', '제60조'),
      ('일하다 위험하면 멈춰도 되나', '산업안전보건법', '제26조'),
      ('직원 정보를 다 쓴 뒤에는', '개인정보보호법', '제21조'),
      ('회사 기술을 외국에 팔면', '산업기술보호법', '제11조')]

def rank_of(sim, src, art):
    # 점수 높은 순으로 줄을 세우고, 정답 조문이 몇 번째인지 센다
    for r, i in enumerate(sim.argsort()[::-1], 1):
        if CHUNKS[i]['source'] == src and CHUNKS[i]['title'].startswith(art):
            return r
    return None

def compare(mat, model):
    print('%-22s %-9s %8s %8s' % ('질문', '정답 조문', '낱말', '의미'))
    for q, src, art in QS:
        t = rank_of(cosine_similarity(vec.transform([q]), M)[0], src, art)
        e = rank_of(mat @ model.encode([q], normalize_embeddings=True)[0], src, art)
        print('%-22s %-9s %7s등 %7s등' % (q, art, t, e))

## 5. 모델을 바꾸면

In [ ]:
# 한국어 문장으로 학습한 모델. 조금 더 크고 조금 더 오래 걸린다.
KO = SentenceTransformer('jhgan/ko-sroberta-multitask')
VK = build(KO)
print('조각 %d개 × %d차원' % VK.shape)

In [ ]:
# 더 나은 쪽으로 갈아 끼운다. 아래부터는 한국어 모델로 찾는다.
EMB, V = KO, VK
print('검색기를 한국어 모델로 바꿨다')

In [ ]:
# 검색에 쓰는 모델이 낱말을 어떻게 쪼개는지 직접 본다
from transformers import AutoTokenizer
tk = AutoTokenizer.from_pretrained('jhgan/ko-sroberta-multitask')

for w in ['휴가', '연차', '제60조', '제61조', 'ERR-5010']:
    t = tk.tokenize(w)
    print('%-10s %d조각  %s' % (w, len(t), t))

In [ ]:
# 현장 용어 사전 — 코드 옆에 무슨 뜻인지 적어 둔다
GLOSSARY = {
    '제60조': '연차 유급휴가',
    'NCM811': '니켈 코발트 망간 양극재',
    'SEI층': '전극 표면 피막',
    '캘린더링': '전극 압연 공정',
}

def expand(q):
    for k, v in GLOSSARY.items():
        if k in q:
            q = q + ' ' + v          # 질문 뒤에 뜻을 덧붙인다
    return q

print(expand('제60조'))

## 6. 더 큰 모델 — API 와 GPU

In [ ]:
# 임베딩도 같은 키로 부른다. 주소만 다르다.
EMB_URL = 'https://integrate.api.nvidia.com/v1/embeddings'

def api_embed(model, texts, kind='passage'):
    body = json.dumps({'model': model, 'input': texts, 'input_type': kind,
                       'encoding_format': 'float', 'truncate': 'END'}).encode()
    req = urllib.request.Request(EMB_URL, data=body, headers={
        'Authorization': 'Bearer ' + KEY,
        'Content-Type': 'application/json', 'Accept': 'application/json'})
    with urllib.request.urlopen(req, timeout=180) as f:
        return [d['embedding'] for d in json.load(f)['data']]

def api_index(model):
    out = []
    for i in range(0, len(TEXTS), 32):          # 한 번에 서른두 개씩
        out += api_embed(model, TEXTS[i:i+32])
    v = np.array(out)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

print(len(api_embed('nvidia/nv-embedqa-e5-v5', ['시험'], 'query')[0]), '차원')

In [ ]:
# 순위를 재는 자 — 모델을 바꿔 가며 같은 다섯 질문을 던진다
def ranks(qvec_fn, mat, name):
    print('%-22s %-9s %8s' % ('질문', '정답 조문', name))
    for q, src, art in QS:
        r = rank_of(mat @ qvec_fn(q), src, art)
        print('%-22s %-9s %7s등' % (q, art, r))

def api_ranks(model):
    V2 = api_index(model)
    qv = lambda q: (lambda v: np.array(v) / np.linalg.norm(v))(
        api_embed(model, [q], 'query')[0])
    ranks(qv, V2, model.split('/')[-1][:8])

In [ ]:
# GPU 가 잡혔는지 먼저 본다
import torch
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('장치', DEV, '·', torch.cuda.get_device_name(0) if DEV == 'cuda' else 'CPU 로 돌아간다')

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 1.** API 모델을 하나 더 골라 같은 다섯 질문을 던진다.
> `nvidia/llama-nemotron-embed-1b-v2` · `nvidia/nv-embed-v1` 중에 골라 본다.
> 위 표에 한 줄을 더 붙인다고 생각하면 된다.

In [ ]:
# 모델 이름만 바꾸면 된다
API_MODEL = '___'

api_ranks(API_MODEL)

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 1.** 세 자리에서 **무엇을 고를지** 정해서 표로 적는다.
> ① 사내 규정 문서로 사내용 검색을 만든다
> ② 공개된 기술 문서로 사외 서비스를 만든다
> ③ 고객 문의 로그로 내부 분석을 한다
> 각각 **노트북 CPU · GPU · API** 중 무엇을 고르고 왜 그런지 한 줄씩.

In [ ]:
ANSWER = {
    '사내 규정': '___ 를 쓴다. 왜냐하면 ___',
    '공개 기술 문서': '___ 를 쓴다. 왜냐하면 ___',
    '고객 문의 로그': '___ 를 쓴다. 왜냐하면 ___',
}
for k, v in ANSWER.items():
    print('%-12s %s' % (k, v))
print('성능보다 먼저 정하는 것은 문서가 나가도 되느냐다')

## 7. 메타데이터로 범위 좁히기

In [ ]:
# 범위를 좁혀서 찾는다. where 에 맞는 조각만 후보로 둔다.
def find3(question, k=3, where=None):
    q = EMB.encode([question], normalize_embeddings=True)[0]
    sim = V @ q
    idx = [i for i, c in enumerate(CHUNKS)
           if not where or all(c[key] == val for key, val in where.items())]
    idx.sort(key=lambda i: -sim[i])
    return [(sim[i], CHUNKS[i]) for i in idx[:k]]

def show3(question, k=3, where=None):
    print('Q %s   %s' % (question, where or '전체'))
    for s, c in find3(question, k, where):
        print('  %.3f [%s] %s' % (s, c['source'], c['title']))
    print()

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 2.** **여러 문서에 다 있을 법한 질문**을 하나 만들어 전체와 좁힘을 견준다.
> 「벌금」 「교육」 「신고」 처럼 어느 법에나 나오는 말이 좋다.

In [ ]:
# where 에 문서 이름을 넣으면 그 문서 안에서만 찾는다
Q4 = '___'
SRC = '___'

show3(Q4)
show3(Q4, where={'source': SRC})

## 8. 등급과 버전으로 거르기

In [ ]:
# 조각마다 등급과 시행일을 붙인다. 사내에서는 문서함의 값을 그대로 옮긴다.
GRADE = {'근로기준법': 'general', '산업안전보건법': 'general',
         '개인정보보호법': 'manager', '산업기술보호법': 'restricted'}
ORDER = {'general': 0, 'manager': 1, 'restricted': 2}       # 낮을수록 널리 열람

for c in CHUNKS:
    c['clearance'] = GRADE[c['source']]
    c['effective_from'] = '2024-01-01'
    c['superseded'] = False

print({g: sum(1 for c in CHUNKS if c['clearance'] == g) for g in ORDER})

In [ ]:
# 개정 전 조항이 인덱스에 남아 있는 상황을 만든다 (내용은 연습용으로 지어낸 것)
OLD = dict(source='근로기준법', chapter='제4장 근로시간과 휴식', article=60,
           title='제60조(연차 유급휴가) [2019년 판]',
           text='제60조(연차 유급휴가) [2019년 판] ① 사용자는 1년간 80퍼센트 이상 출근한 '
                '근로자에게 10일의 유급휴가를 주어야 한다.',
           clearance='general', effective_from='2019-01-01', superseded=True)
CHUNKS.append(OLD)
V = np.vstack([V, EMB.encode([OLD['title'] + ' ' + OLD['text']],
                             normalize_embeddings=True)])
print('조각 %d개 — 구버전 하나가 섞였다' % len(CHUNKS))

In [ ]:
# 찾기 전에 거른다. 등급과 시행일을 후보 단계에서 잘라 낸다.
def find4(question, k=3, grade='general', include_old=False):
    q = EMB.encode([question], normalize_embeddings=True)[0]
    sim = V @ q
    idx = [i for i, c in enumerate(CHUNKS)
           if ORDER[c['clearance']] <= ORDER[grade]                 # 등급
           and (include_old or not c['superseded'])]                # 버전
    idx.sort(key=lambda i: -sim[i])
    return [(sim[i], CHUNKS[i]) for i in idx[:k]]

def show4(question, k=3, grade='general'):
    print('Q %s   등급 %s' % (question, grade))
    for s, c in find4(question, k, grade):
        print('  %.3f [%-8s %s] %s' % (s, c['clearance'], c['source'], c['title']))
    print()

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 3.** `grade` 를 **manager** 로 두고 개인정보 질문을 던진다.
> 일반 등급으로 물었을 때와 무엇이 달라지는지 본다.

In [ ]:
# 개인정보보호법은 manager 등급으로 붙여 두었다
GRADE_TEST = '___'

show4('개인정보를 다 쓰고 나면 어떻게 하나', grade='general')
show4('개인정보를 다 쓰고 나면 어떻게 하나', grade=GRADE_TEST)

## 9. 근거로만 답하게 하기

In [ ]:
# 찾은 조각을 붙여 물어보는 함수. RAG 는 이 열 줄이 전부다.
RULE = ('아래 「문서」에 있는 내용만 근거로 답하라.\n'
        '문서에 없으면 "문서에 없다"고만 답하라. 아는 것으로 채우지 마라.\n'
        '답 끝에 근거로 쓴 조문 제목과 시행일을 적어라.\n'
        '근거를 못 대면 답하지 마라.\n\n')

def rag(question, k=3, grade='general', log=True):
    hits = find4(question, k, grade)          # 등급·버전을 먼저 거른 뒤 찾는다
    ctx = '\n\n'.join('[%s %s | 시행 %s] %s'
                       % (c['source'], c['title'][:24], c['effective_from'], c['text'][:700])
                       for _, c in hits)
    if log:
        for s, c in hits:
            print('  [찾음 %.3f] %s' % (s, c['title']))
    return ask(RULE + '# 문서\n' + ctx + '\n\n# 질문\n' + question)

## 11. 내 문서로

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 2.** 자기 부서 문서를 `.txt` 로 만들어 올리고 같은 흐름을 돌린다.
> ① Colab 왼쪽 파일 창에 `.txt` 를 올린다 → ② 아래에서 읽는다 →
> ③ 자르고 → ④ 좌표로 바꾸고 → ⑤ 물어본다. **코드는 안 고친다.**
> 문서를 자르는 규칙만 자기 문서에 맞게 정한다.

In [ ]:
MY = open('___.txt', encoding='utf-8').read()
MY_CHUNKS = split_articles(MY, '내 문서')      # 「제N조」 규칙이 안 맞으면 아래 Task 2 로
print('조각', len(MY_CHUNKS), '개')
print(MY_CHUNKS[0]['title'])

> **실습문제 3.** 자를 자리를 **자기 문서 규칙**으로 바꾼다.
> 사내 문서는 「제N조」가 아니라 「1.」 「가.」 「■」 같은 표시로 나뉜다.
> 정규식 한 줄만 바꾸면 된다. 자른 뒤 **조각 하나만 읽어서 말이 되는지** 본다.

In [ ]:
PATTERN = r'___'
parts = [p.strip() for p in re.split(PATTERN, MY) if len(p.strip()) > 40]
print('조각 %d개 · 첫 조각' % len(parts))
print(parts[0][:120])
print('조각 하나가 맥락 없이 읽혀야 검색이 산다')

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 4.** 자기 문서로 **답이 틀리는 질문**을 하나 찾아 온다.
> 틀렸을 때 원인이 셋 중 어디인지 갈라 본다.
> ① 검색이 엉뚱한 조각을 가져왔나 → 자르는 자리·k 를 손본다
> ② 가져왔는데 답이 틀렸나 → 프롬프트 규칙을 손본다
> ③ 문서에 아예 없나 → 「문서에 없다」가 나오면 정상이다

In [ ]:
BAD_Q = '___'
print(rag(BAD_Q))
print('찾은 조각 제목을 먼저 본다. 거기서 원인이 갈린다')

> **실습문제 5.** **Codex 에 시킬 프롬프트**를 쓴다. 코드가 아니라 **조건**을 적는 연습이다.
> 아래 네 가지는 사람이 정해서 적어 줘야 한다. 안 적으면 지어낸다.
> ① 인덱싱에서 뺄 문서 ② 등급 규칙 ③ 자르는 단위 ④ 임베딩·저장·생성이 각각 어디서
> 특히 **「등급 필터는 검색 전에 건다」**를 안 적으면 검색 뒤에 거르는 코드가 온다.

In [ ]:
MY_PROMPT = '\n'.join([
    '# 역할', '너는 사내 규정 검색 코드를 쓰는 사람이다.', '',
    '# 넣지 않을 문서', '___', '',
    '# 등급 규칙', '___', '',
    '# 자르는 단위', '___', '',
    '# 어디서 도나', '임베딩 ___ · 벡터 저장 ___ · 생성 ___', '',
    '# 형식', '바로 돌아가는 파이썬 코드로 준다. 문서 내용을 출력하는 줄은 넣지 마라.'])
print(MY_PROMPT)
print('코드는 맡기고 기준은 맡기지 않는다')